In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/anime_reviews.csv")

In [3]:
## Recreating Sentiment Column
def score_to_sentiment(score):
    if score >= 8:
        return "Positive"
    elif score >= 5:
        return "Neutral"
    return "Negative"

df["sentiment"] = df["score"].apply(score_to_sentiment)

In [4]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/manaskolaskar/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/manaskolaskar/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/manaskolaskar/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/manaskolaskar/nltk_data...


True

In [5]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

In [6]:
sample = df["review_text"].iloc[0]

print("ORIGINAL:")
print(sample[:300])

print("\nCLEANED:")
print(clean_text(sample[:300]))

ORIGINAL:
Makoto Shinkai's The Garden of Words is one of those films that earns the word "immersive" without having to ask for it. From the first scene, it pulls you in — not with action, not with urgency, but with the quiet insistence of rain on glass. You settle into it before you've even decided to. The vi

CLEANED:
makoto shinkais the garden of words is one of those films that earns the word immersive without having to ask for it from the first scene it pulls you in  not with action not with urgency but with the quiet insistence of rain on glass you settle into it before youve even decided to the vi


In [10]:
## Tokenization
import nltk

print(nltk.data.path)

['/Users/manaskolaskar/nltk_data', '/Users/manaskolaskar/Developer/Projects/otaku-ai/.venv/nltk_data', '/Users/manaskolaskar/Developer/Projects/otaku-ai/.venv/share/nltk_data', '/Users/manaskolaskar/Developer/Projects/otaku-ai/.venv/lib/nltk_data', '/usr/share/nltk_data', '/usr/local/share/nltk_data', '/usr/lib/nltk_data', '/usr/local/lib/nltk_data']


In [12]:
from nltk.tokenize import word_tokenize

sample_cleaned = clean_text(df["review_text"].iloc[0])

tokens = word_tokenize(sample_cleaned)

print(tokens[:50])

['makoto', 'shinkais', 'the', 'garden', 'of', 'words', 'is', 'one', 'of', 'those', 'films', 'that', 'earns', 'the', 'word', 'immersive', 'without', 'having', 'to', 'ask', 'for', 'it', 'from', 'the', 'first', 'scene', 'it', 'pulls', 'you', 'in', 'not', 'with', 'action', 'not', 'with', 'urgency', 'but', 'with', 'the', 'quiet', 'insistence', 'of', 'rain', 'on', 'glass', 'you', 'settle', 'into', 'it', 'before']


In [13]:
## StopWords Removal
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

filtered_tokens = [
    word for word in tokens
    if word not in stop_words
]

print(filtered_tokens[:50])

['makoto', 'shinkais', 'garden', 'words', 'one', 'films', 'earns', 'word', 'immersive', 'without', 'ask', 'first', 'scene', 'pulls', 'action', 'urgency', 'quiet', 'insistence', 'rain', 'glass', 'settle', 'youve', 'even', 'decided', 'visuals', 'hit', 'first', 'hit', 'hard', 'every', 'backdrop', 'wet', 'garden', 'tiles', 'light', 'fracturing', 'leaves', 'particular', 'grey', 'tokyo', 'sky', 'open', 'looks', 'less', 'like', 'animation', 'like', 'memory', 'character', 'designs']


In [14]:
## lemmatization
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

lemmatized_tokens = [
    lemmatizer.lemmatize(word)
    for word in filtered_tokens
]

print(lemmatized_tokens[:50])

['makoto', 'shinkais', 'garden', 'word', 'one', 'film', 'earns', 'word', 'immersive', 'without', 'ask', 'first', 'scene', 'pull', 'action', 'urgency', 'quiet', 'insistence', 'rain', 'glass', 'settle', 'youve', 'even', 'decided', 'visuals', 'hit', 'first', 'hit', 'hard', 'every', 'backdrop', 'wet', 'garden', 'tile', 'light', 'fracturing', 'leaf', 'particular', 'grey', 'tokyo', 'sky', 'open', 'look', 'less', 'like', 'animation', 'like', 'memory', 'character', 'design']


In [15]:
def preprocess_text(text):
    text = clean_text(text)

    tokens = word_tokenize(text)

    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [16]:
sample = df["review_text"].iloc[0]

processed = preprocess_text(sample)

print(processed[:500])

makoto shinkais garden word one film earns word immersive without ask first scene pull action urgency quiet insistence rain glass settle youve even decided visuals hit first hit hard every backdrop wet garden tile light fracturing leaf particular grey tokyo sky open look less like animation like memory character design precise without sterile there tactile quality everyscene feeling reached screen youd come back damp finger kind craft make pause midscene sit youre looking knowing came almost fee


In [17]:
df["processed_review"] = df["review_text"].apply(preprocess_text)

In [18]:
df[["review_text", "processed_review"]].head(3)

,review_text,processed_review
0,Makoto Shinkai's The Garden of Words is one of...,makoto shinkais garden word one film earns wor...
1,Y con es termino de ver todas las peliculas de...,con e termino de ver toda la peliculas de mi d...
2,My honest opinion on K-ON!: I absolutely LOVE ...,honest opinion kon absolutely love show yui hi...


In [19]:
## TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

In [20]:
tfidf = TfidfVectorizer(max_features=1000)

X = tfidf.fit_transform(df["processed_review"])

In [21]:
print(type(X))
print(X.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(1490, 1000)


In [22]:
feature_names = tfidf.get_feature_names_out()

print(feature_names[:50])

['ability' 'able' 'absolutely' 'academia' 'across' 'act' 'acting' 'action'
 'actor' 'actual' 'actually' 'adaptation' 'adapted' 'add' 'admit'
 'adventure' 'age' 'ago' 'ahead' 'alchemist' 'almost' 'alone' 'along'
 'alot' 'already' 'also' 'although' 'always' 'amazing' 'among' 'amount'
 'amp' 'animated' 'animation' 'anime' 'annoying' 'another' 'answer'
 'antagonist' 'anyone' 'anything' 'anyway' 'aot' 'apart' 'appeal' 'april'
 'arc' 'area' 'arent' 'armin']


In [23]:
y = df["sentiment"]

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [25]:
print(X_train.shape)
print(X_test.shape)

(1192, 1000)
(298, 1000)


In [26]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](3,)","[201.,312.,679.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](3,)","[-1.78,-1.34,-0.56]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U8](3,)","['Negative','Neutral','Positive']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](3, 1000)","[[0.75,1.11,1.19,...,2.59,0.42,1.16], [2.75,1.82,2.38,...,2.15,1.11,0.91], [4.7 ,6.63,6.6 ,...,5.67,3.56,2.48]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](3, 1000)","[[-7.28,-7.09,-7.06,...,-6.56,-7.49,-7.07], [-6.82,-7.1 ,-6.92,...,-6.99,-7.39,-7.49], [-6.98,-6.69,-6.69,...,-6.82,-7.2 ,-7.47]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1000


In [27]:
y_pred = nb_model.predict(X_test)

print(y_pred[:10])

['Positive' 'Positive' 'Positive' 'Positive' 'Positive' 'Positive'
 'Positive' 'Positive' 'Neutral' 'Negative']


In [29]:
### Evaluation
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.6141


In [30]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    Negative       0.80      0.08      0.15        50
     Neutral       0.48      0.13      0.20        78
    Positive       0.62      0.99      0.76       170

    accuracy                           0.61       298
   macro avg       0.63      0.40      0.37       298
weighted avg       0.61      0.61      0.51       298



In [31]:
import pandas as pd

pd.Series(y_pred).value_counts()

Positive    272
Neutral      21
Negative      5
Name: count, dtype: int64

In [32]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[  4  10  36]
 [  1  10  67]
 [  0   1 169]]


In [33]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

In [34]:
from sklearn.metrics import accuracy_score

print(accuracy_score(y_test, y_pred_lr))

0.6375838926174496


In [35]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

    Negative       0.75      0.18      0.29        50
     Neutral       0.40      0.23      0.29        78
    Positive       0.68      0.96      0.79       170

    accuracy                           0.64       298
   macro avg       0.61      0.46      0.46       298
weighted avg       0.62      0.64      0.58       298



In [36]:
feature_names = tfidf.get_feature_names_out()

In [37]:
import pandas as pd

positive_class_idx = list(lr_model.classes_).index("Positive")

coef_df = pd.DataFrame({
    "word": feature_names,
    "weight": lr_model.coef_[positive_class_idx]
})

coef_df.sort_values("weight", ascending=False).head(20)

,word,weight
646,perfect,1.424940
34,anime,1.252884
823,steinsgate,1.187634
521,love,1.157958
750,season,1.142637
376,great,1.114947
943,voice,1.109062
28,amazing,1.072005
590,naruto,1.062224
978,world,0.957365


In [38]:
negative_class_idx = list(lr_model.classes_).index("Negative")

negative_df = pd.DataFrame({
    "word": feature_names,
    "weight": lr_model.coef_[negative_class_idx]
})

negative_df.sort_values(
    "weight",
    ascending=False
).head(20)

,word,weight
474,kirito,1.307297
171,could,1.194267
365,girl,1.161292
952,waste,1.146004
104,boring,1.029219
852,supposed,1.001846
608,nothing,0.946826
289,extremely,0.926620
979,worse,0.910189
740,sao,0.877245


In [39]:
binary_df = df.copy()

binary_df = binary_df[
    ~binary_df["score"].isin([6, 7])
]

binary_df["binary_sentiment"] = binary_df["score"].apply(
    lambda x: "Positive" if x >= 8 else "Negative"
)

binary_df["binary_sentiment"].value_counts()

binary_sentiment
Positive    849
Negative    383
Name: count, dtype: int64

In [40]:
X_binary = binary_df["processed_review"]

y_binary = binary_df["binary_sentiment"]

In [41]:
## New TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_binary = TfidfVectorizer(
    max_features=1000
)

X_binary = tfidf_binary.fit_transform(
    X_binary
)

In [42]:
from sklearn.model_selection import train_test_split

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_binary,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

In [43]:
from sklearn.linear_model import LogisticRegression

lr_binary = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_binary.fit(
    X_train_b,
    y_train_b
)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [44]:
y_pred_b = lr_binary.predict(
    X_test_b
)

In [45]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

print(
    f"Accuracy: {accuracy_score(y_test_b, y_pred_b):.4f}"
)

print(
    classification_report(
        y_test_b,
        y_pred_b
    )
)

Accuracy: 0.7733
              precision    recall  f1-score   support

    Negative       0.86      0.32      0.47        77
    Positive       0.76      0.98      0.86       170

    accuracy                           0.77       247
   macro avg       0.81      0.65      0.66       247
weighted avg       0.79      0.77      0.74       247



In [46]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test_b,
    y_pred_b
)

print(cm)

[[ 25  52]
 [  4 166]]


In [47]:
from sklearn.linear_model import LogisticRegression

lr_balanced = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

lr_balanced.fit(
    X_train_b,
    y_train_b
)

y_pred_balanced = lr_balanced.predict(
    X_test_b
)

In [48]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print(
    f"Accuracy: {accuracy_score(y_test_b, y_pred_balanced):.4f}"
)

print(
    classification_report(
        y_test_b,
        y_pred_balanced
    )
)

print(
    confusion_matrix(
        y_test_b,
        y_pred_balanced
    )
)

Accuracy: 0.8259
              precision    recall  f1-score   support

    Negative       0.69      0.81      0.74        77
    Positive       0.90      0.84      0.87       170

    accuracy                           0.83       247
   macro avg       0.80      0.82      0.81       247
weighted avg       0.84      0.83      0.83       247

[[ 62  15]
 [ 28 142]]


In [49]:
import joblib

In [50]:
joblib.dump(
    lr_balanced,
    "../artifacts/sentiment_model.pkl"
)

joblib.dump(
    tfidf_binary,
    "../artifacts/tfidf_vectorizer.pkl"
)

['../artifacts/tfidf_vectorizer.pkl']

In [51]:
import os

print(os.listdir("../artifacts"))

['tfidf_vectorizer.pkl', 'sentiment_model.pkl']


In [52]:
print(df.columns.tolist())

['review_id', 'anime_mal_id', 'anime_title', 'username', 'score', 'tags', 'review_text', 'date', 'episodes_watched', 'is_spoiler', 'is_preliminary', 'reactions_overall', 'reactions_nice', 'reactions_love_it', 'reactions_funny', 'reactions_confusing', 'reactions_informative', 'reactions_well_written', 'reactions_creative', 'sentiment', 'processed_review']


In [53]:
"binary_sentiment" in df.columns

False